# Gold: patient_360

Builds `<catalog>.gold.patient_360` per the Silver-to-Gold STTM: one row per patient, driven by `silver.patients` with LEFT JOINs to hospitals, insurance providers and aggregated lab results. Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);

CREATE OR REPLACE TABLE gold.patient_360
COMMENT 'One row per patient: demographics, registered hospital, insurance provider and lab test summary'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                   AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)     AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)     AS critical_test_count,
    MAX(test_date)                                             AS last_test_date
  FROM silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                              AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                   AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                                                           AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                    AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                            AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                            AS critical_test_count,
  l.last_test_date
FROM silver.patients p
LEFT JOIN silver.hospitals h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN silver.insurance_providers ip ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l                     ON p.patient_id = l.patient_id;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM silver.patients)  AS source_patients,
  COUNT(*)                                AS gold_rows,
  COUNT(DISTINCT patient_id)              AS distinct_patients,
  SUM(total_tests)                        AS total_tests,
  (SELECT COUNT(*) FROM silver.lab_results) AS source_lab_results,
  SUM(CASE WHEN hospital_name IS NULL THEN 1 ELSE 0 END)           AS missing_hospital,
  SUM(CASE WHEN insurance_provider_name IS NULL THEN 1 ELSE 0 END) AS missing_insurance
FROM gold.patient_360